# MUKHI on Kaggle

Script or language? Punjabi Gurmukhi / Shahmukhi as a minimal pair.

The GPU quota is 30 hours a week, and most of this pipeline is CPU work (the
permutation tests, about 6 hours in total). So the run is split into three
sessions:

| session | accelerator | stages | rough time |
|---|---|---|---|
| A | none (CPU) | setup, tests, translit_eval, g2s_compare, conditions, fertility, audit | under 1 h |
| B | GPU T4 x2 | extract activations for every model; intervention stages | 3 to 4 h |
| C | none (CPU) | analysis from saved activations, report | about 6 h |

Session B saves activations to `/kaggle/working/acts`. Save the notebook
output as a Kaggle Dataset, then attach that dataset in session C.

**Before starting**, on huggingface.co with the account whose token you use:
accept the terms for `openlanguagedata/flores_plus`, `google/gemma-2-2b` and
`CohereLabs/aya-expanse-8b`. Then add the token to Kaggle as a secret named
`HF_TOKEN` (Add-ons, Secrets).

## Setup (every session)

In [ ]:
!pip -q install uroman 2>/dev/null
import os, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("No HF token secret:", e)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpus", torch.cuda.device_count())

In [ ]:
import os, glob, shutil, tarfile
ROOT = "/kaggle/working/mukhi"
CACHE = "/kaggle/working/hf_cache"
if not os.path.exists(os.path.join(ROOT, "run_all.py")):
    hits = glob.glob("/kaggle/input/**/run_all.py", recursive=True)
    if hits:
        shutil.copytree(os.path.dirname(hits[0]), ROOT, dirs_exist_ok=True)
    else:
        tars = glob.glob("/kaggle/input/**/mukhi*.tar.gz", recursive=True)
        assert tars, "No run_all.py or mukhi.tar.gz under /kaggle/input: attach the dataset via Add Input"
        tarfile.open(tars[0]).extractall("/kaggle/working")
os.makedirs(CACHE, exist_ok=True)
os.chdir(ROOT)
print("working in", os.getcwd(), "| run_all.py found:", os.path.exists("run_all.py"))


## Session A (CPU)

### A1. Transliteration quality on human text (PuMVR)

Fetches the PuMVR JSON from GitHub at a pinned commit (1.5 MB). Expect
S->G context rules about 0.16 word CER and G->S rules about 0.06.

In [ ]:
!python run_all.py --stage translit_eval

### A2. Tests

Run after A1, so the transliteration regression tests find the PuMVR data.
Every numerical component is checked against planted ground truth. Run once
per environment: numpy and sklearn versions differ from the development box.

In [ ]:
!for t in tests/test_*.py; do printf "%-24s " $(basename $t); python3 $t 2>&1 | tail -1; done

In [ ]:
!python run_all.py --stage smoke

### A3. Optional: SLPG fairseq model

Only replaces the rules if it scores better on PuMVR. fairseq often fails to
install on Python 3.11; if so this cell just reports it and the rules are used.

In [ ]:
!pip -q install fairseq==0.12.2 2>&1 | tail -2
!python run_all.py --stage g2s_compare --cache-dir $CACHE

### A4. Build all conditions (FLORES+ and SIB-200)

FLORES+ has no Shahmukhi and no Serbian Latin: both are generated here, once,
and cached so later stages can never silently differ.

In [ ]:
!python run_all.py --stage conditions --cache-dir $CACHE

### A5. Tokeniser premium (no GPU)

Tokenisers only. Watch `byte_fallback_rate`. The direction of the premium is
an open question, not an assumption.

In [ ]:
!python run_all.py --stage fertility --cache-dir $CACHE
!python run_all.py --stage fertility --data pumvr --cache-dir $CACHE

### A6. Audit sheet

300 generated-Shahmukhi rows; 60% flagged by the independent round-trip
check, 40% random. Download `results/audit_sheet.csv`, judge the Shahmukhi
column (ok / error, plus error_type), re-upload, then run the second cell.

In [ ]:
!python run_all.py --stage audit

In [ ]:
# after filling the sheet:
# !python run_all.py --stage score_audit

**End of session A.** Save Version (Save and Run All is not needed; a quick
save keeps `results/`). Attach this output to session B.

## Session B (GPU T4 x2)

### B1. Extract activations, trained models

Extraction only; analysis happens on CPU in session C. Resumable: an existing
`.npy` is reused.

In [ ]:
# bring in session A outputs (attach the session A notebook output via Add Input)
src = next((h for h in glob.glob("/kaggle/input/**/results/conditions_flores.json", recursive=True)), None)
assert src, "Attach the session A output (it contains results/conditions_flores.json)"
!cp -rn {os.path.dirname(src)}/* results/
!ls results | head
ACTS = "/kaggle/working/acts"
!python run_all.py --stage main --extract-only --acts-dir $ACTS --cache-dir $CACHE
!python run_all.py --stage main --data pumvr --extract-only --acts-dir $ACTS --cache-dir $CACHE
!du -sh $ACTS

### B2. Random-init baselines (FINDING_07)

Same architectures, random weights, three seeds. Aya is skipped: initialising
it in fp32 needs about 32 GB of host memory.

In [ ]:
SMALL = "xlm-roberta-base,google/muril-base-cased,xlm-roberta-large,Qwen/Qwen2.5-1.5B"
for seed in (0, 1, 2):
    !python run_all.py --stage main --extract-only --acts-dir $ACTS --random-init --seed $seed --models $SMALL --cache-dir $CACHE

### B3. Intervention (proposal step 5), both arms

In [ ]:
!python run_all.py --stage intervention --cache-dir $CACHE
!python run_all.py --stage intervention_mcq --cache-dir $CACHE

**End of session B.** Save Version, then create a Dataset from this notebook's
output (it contains `acts/` and `results/`).

## Session C (CPU)

Attach the session B output dataset. Activations are read from it; nothing is
re-extracted.

In [ ]:
src = next((h for h in glob.glob("/kaggle/input/**/acts/*.npy", recursive=True)), None)
assert src, "Attach the session B output (it contains acts/*.npy)"
ACTS_IN = os.path.dirname(src)
res = os.path.join(os.path.dirname(ACTS_IN), "results")
if os.path.isdir(res):
    !cp -rn {res}/* results/
print("activations from", ACTS_IN)
!python run_all.py --stage main --acts-dir $ACTS_IN --force
!python run_all.py --stage main --data pumvr --acts-dir $ACTS_IN --force
SMALL = "xlm-roberta-base,google/muril-base-cased,xlm-roberta-large,Qwen/Qwen2.5-1.5B"
for seed in (0, 1, 2):
    !python run_all.py --stage main --acts-dir $ACTS_IN --random-init --seed $seed --models $SMALL --force

In [ ]:
!python run_all.py --stage report
from IPython.display import Markdown, display
display(Markdown(open("results/REPORT.md").read()))

## Reading the results

- The depth claim rests on the **paired displacement** minus its random-init
  baseline, not on probe accuracy (FINDING_02, FINDING_07).
- `pan_Guru vs pan_Guru_devowel` is the information part of the script gap;
  `pan_Guru_devowel vs pan_Shah` is the script-shape part (FINDING_04).
- `srp_Cyrl vs srp_Latn` is the balanced, lossless contrast. If Serbian and
  Punjabi disagree, the asymmetries are the likely reason.
- In the intervention table, if Shah->Guru lands near Guru devowel, the
  remaining gap is information Shahmukhi never had, not a bottleneck any
  transliteration could fix.